# Capstone — Refresh / Content Opportunity Scoring\n\nThis notebook mirrors the deployed paper and produces a ranked queue of refresh opportunities using the anonymized FlyRank starter dataset.

## 1. Question\n**Decision supported:** which pages should be reviewed first for refresh work.\n\nWe frame this as ranking page-level opportunities by decline risk and recovery potential, then map the top pages to concrete editor actions.

In [ ]:
from pathlib import Path\nimport subprocess, sys\n\nROOT = Path.cwd()\nif not (ROOT / 'scripts').exists():\n    ROOT = ROOT.parent.parent\n\nprint('Repo root:', ROOT)\nsubprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_all.py')], check=True)\nprint('Pipeline run complete.')

## 2. Data\n- Source: `data/raw/content_refresh_anonymized.csv` (30,000 rows, pseudonymized).\n- Unit: one row per content item over a trailing 90-day window.\n- Public-safe exclusions: no client names/domains/URLs/raw queries/credentials.\n- Label leakage controls: `trend_direction` and `trend_pct` are excluded as model features because the label derives from them.

In [ ]:
import pandas as pd\n\nraw = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')\nprint('Rows:', len(raw))\nprint('Columns:', raw.shape[1])\nprint('Clients:', raw['client_id'].nunique())\nprint('Declining label rate:', round((raw['trend_direction'] == 'down').mean(), 3))

## 3. Methodology\n- Lane: **Refresh / Content Opportunity Scoring**.\n- Baseline: transparent rules engine (`02_baseline_score.py`) with reason codes.\n- Model: logistic regression, decision tree, random forest (`03_train_model.py`).\n- Validation: client-holdout split to test cross-client generalization.\n- Leakage checks: no IDs or label-derived fields used as predictors.

In [ ]:
report = (ROOT / 'outputs' / 'model_report.md').read_text()\nfor line in report.splitlines():\n    if '| random_forest |' in line or '| baseline_rules |' in line:\n        print(line)

## 4. Results (vs baseline)\nOn the same validation split, the selected random forest reaches **Precision@50 = 0.740**, while the baseline rules score **0.240**.\n\nThis is an observed lift for ranking quality in this dataset and split; it is decision support, not a causal claim about Google ranking behavior.

In [ ]:
queue_path = ROOT / 'outputs' / 'refresh_queue.csv'\nif not queue_path.exists():\n    queue_path = ROOT / 'outputs' / 'refresh_queue_sample.csv'\n\nqueue = pd.read_csv(queue_path)\nprint('Queue source:', queue_path.name)\nprint('Rows ranked:', len(queue))\nprint('Top suggested actions:')\nprint(queue['suggested_action'].value_counts().head(5).to_string())\ndisplay(queue[['final_rank','final_refresh_score','best_model_probability','suggested_action','final_reason_codes']].head(10))

## 5. Limitations\n- This analysis is observational and does not prove causality.\n- Results come from a starter anonymized slice; full-warehouse replication is still needed for production use.\n- Recommendations are triage aids and require human editorial review before execution.

In [ ]:
print('Limitation check: interpretation language stays directional / decision-support.')

## 6. Ranked recommendations\n1. **Refresh + CTR review first** for high-visibility, low-CTR pages with decline signals.\n2. **Refresh + engagement review** where scroll/session behavior is weak despite visibility.\n3. **Monitor only** lower-confidence pages to avoid over-editing low-evidence items.

In [ ]:
top_cols = ['final_rank','content_id','client_id','final_refresh_score','suggested_action','final_reason_codes']\ndisplay(queue[top_cols].head(20))

## 7. Reproducibility artifacts\n- Pipeline command: `python scripts/run_all.py`\n- Report: `outputs/model_report.md`\n- Queue CSV: `outputs/refresh_queue.csv` (generated)\n- Example queue in git: `outputs/refresh_queue_sample.csv`\n- Paper source: `docs/index.html`\n- Data credit: **Built on the FlyRank ML Internship dataset** (https://flyrank.ai)

In [ ]:
artifact_paths = [\n    ROOT / 'outputs' / 'model_report.md',\n    ROOT / 'outputs' / 'charts' / 'top_feature_importance.svg',\n    ROOT / 'outputs' / 'charts' / 'action_mix.svg',\n]\nfor p in artifact_paths:\n    print(p.name, 'exists' if p.exists() else 'missing')

## Self-check\n- [x] Model compared against baseline on same split\n- [x] Leakage risks addressed\n- [x] Ranked recommendations produced\n- [x] Public-safe language and anonymized data usage only\n- [x] Paper includes required FlyRank data credit